# Analyse statistique — Facteurs associés à l'attrition

Ce notebook vise à vérifier statistiquement si certaines variables semblent associées au départ des employés.

Les analyses portent notamment sur :

- les heures supplémentaires ;
- le département ;
- le poste ;
- le salaire ;
- l'âge ;
- l'ancienneté ;
- la satisfaction au travail.

L'objectif n'est pas de prouver une causalité, mais d'identifier les variables fortement associées à l'attrition.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind

In [2]:
data_path = Path("../data/processed/employee_attrition_clean.csv")
df = pd.read_csv(data_path)
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition_Flag
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,1,0,8,0,1,6,4,0,5,1
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,1,10,3,3,10,7,1,7,0
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,2,0,7,3,3,0,0,0,0,1
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,0,8,3,3,8,7,3,0,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,4,1,6,3,3,2,2,2,2,0


In [3]:
df.shape

(1470, 32)

## Comprendre la p-value

Dans cette analyse, on utilise une règle simple :

- si la p-value est inférieure à 0,05, on considère que la relation observée est statistiquement significative ;
- si la p-value est supérieure ou égale à 0,05, on considère que les données ne montrent pas de relation statistiquement significative.

Attention : une relation statistiquement significative ne prouve pas une causalité.  
Elle indique seulement qu'il existe une association observable dans les données.

## 1. Test statistique : heures supplémentaires et attrition

In [4]:
contingency_overtime = pd.crosstab(df["OverTime"], df["Attrition"])
contingency_overtime

Attrition,No,Yes
OverTime,,
No,944,110
Yes,289,127


In [5]:
chi2, p_value, dof, expected = chi2_contingency(contingency_overtime)

print(f"Statistique du khi-deux : {chi2:.2f}")
print(f"p-value : {p_value:.5f}")
print(f"Degrés de liberté : {dof}")

Statistique du khi-deux : 87.56
p-value : 0.00000
Degrés de liberté : 1


### Interprétation

Le test du khi-deux permet de vérifier s'il existe une association statistique entre les heures supplémentaires et l'attrition.

Si la p-value est inférieure à 0,05, on peut considérer que les heures supplémentaires sont significativement associées au départ des employés.

Cette relation ne prouve pas que les heures supplémentaires causent directement les départs, mais elle indique que les employés concernés par les heures supplémentaires quittent plus souvent l'entreprise dans ce dataset.

In [6]:
def chi_square_test(data, variable, target="Attrition"):
    contingency_table = pd.crosstab(data[variable], data[target])
    
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    result = {
        "Variable": variable,
        "Chi2": round(chi2, 2),
        "p_value": round(p_value, 5),
        "Significatif": p_value < 0.05
    }
    
    return result

In [7]:
categorical_variables = [
    "OverTime",
    "Department",
    "JobRole",
    "BusinessTravel",
    "JobSatisfaction",
    "EnvironmentSatisfaction",
    "WorkLifeBalance",
    "MaritalStatus"
]

chi_square_results = []

for variable in categorical_variables:
    result = chi_square_test(df, variable)
    chi_square_results.append(result)

chi_square_results_df = pd.DataFrame(chi_square_results)

chi_square_results_df.sort_values("p_value")

,Variable,Chi2,p_value,Significatif
0,OverTime,87.56,0.00000,True
2,JobRole,86.19,0.00000,True
7,MaritalStatus,46.16,0.00000,True
3,BusinessTravel,24.18,0.00001,True
5,EnvironmentSatisfaction,22.50,0.00005,True
4,JobSatisfaction,17.51,0.00056,True
6,WorkLifeBalance,16.33,0.00097,True
1,Department,10.80,0.00453,True


### Interprétation des tests catégoriels

Les tests du khi-deux permettent d'identifier les variables catégorielles statistiquement associées à l'attrition.

Les variables significatives indiquent que la répartition des départs n'est pas identique selon les catégories observées.

Par exemple, si `OverTime` est significatif, cela signifie que le taux de départ diffère fortement entre les employés qui font des heures supplémentaires et ceux qui n'en font pas.

Ces résultats permettent de prioriser les facteurs à analyser dans le modèle prédictif et dans les recommandations RH.

## 2. Comparaison des variables numériques

In [8]:
numeric_variables = [
    "Age",
    "MonthlyIncome",
    "YearsAtCompany",
    "DistanceFromHome",
    "TotalWorkingYears",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]

In [9]:
def compare_numeric_variable(data, variable, target_flag="Attrition_Flag"):
    stayed = data[data[target_flag] == 0][variable]
    left = data[data[target_flag] == 1][variable]
    
    t_stat, p_value = ttest_ind(stayed, left, equal_var=False)
    
    result = {
        "Variable": variable,
        "Moyenne_restés": round(stayed.mean(), 2),
        "Moyenne_partis": round(left.mean(), 2),
        "Différence": round(left.mean() - stayed.mean(), 2),
        "p_value": round(p_value, 5),
        "Significatif": p_value < 0.05
    }
    
    return result

In [10]:
numeric_results = []

for variable in numeric_variables:
    result = compare_numeric_variable(df, variable)
    numeric_results.append(result)

numeric_results_df = pd.DataFrame(numeric_results)

numeric_results_df.sort_values("p_value")

,Variable,Moyenne_restés,Moyenne_partis,Différence,p_value,Significatif
0,Age,37.56,33.61,-3.95,0.00000,True
1,MonthlyIncome,6832.74,4787.09,-2045.65,0.00000,True
2,YearsAtCompany,7.37,5.13,-2.24,0.00000,True
4,TotalWorkingYears,11.86,8.24,-3.62,0.00000,True
7,YearsWithCurrManager,4.37,2.85,-1.52,0.00000,True
5,YearsInCurrentRole,4.48,2.90,-1.58,0.00000,True
3,DistanceFromHome,8.92,10.63,1.72,0.00414,True
6,YearsSinceLastPromotion,2.23,1.95,-0.29,0.19865,False


### Interprétation des comparaisons numériques

Les tests de comparaison de moyennes permettent de voir si les employés partis présentent des caractéristiques numériques différentes des employés restés.

Par exemple :

- un âge moyen plus faible chez les employés partis peut indiquer une attrition plus forte chez les profils jeunes ;
- une ancienneté plus faible chez les employés partis peut signaler un enjeu d'intégration ou d'onboarding ;
- un revenu mensuel moyen plus faible chez les employés partis peut suggérer que la rémunération joue un rôle, même si cette variable dépend aussi du poste, du niveau hiérarchique et de l'expérience.

Ces résultats doivent être interprétés comme des associations statistiques, et non comme des preuves de causalité.

## 3. Synthèse statistique finale

In [11]:
significant_categorical = chi_square_results_df[
    chi_square_results_df["Significatif"] == True
].sort_values("p_value")

significant_numeric = numeric_results_df[
    numeric_results_df["Significatif"] == True
].sort_values("p_value")

print("Variables catégorielles significatives :")
display(significant_categorical)

print("Variables numériques significatives :")
display(significant_numeric)

Variables catégorielles significatives :


,Variable,Chi2,p_value,Significatif
0,OverTime,87.56,0.00000,True
2,JobRole,86.19,0.00000,True
7,MaritalStatus,46.16,0.00000,True
3,BusinessTravel,24.18,0.00001,True
5,EnvironmentSatisfaction,22.50,0.00005,True
4,JobSatisfaction,17.51,0.00056,True
6,WorkLifeBalance,16.33,0.00097,True
1,Department,10.80,0.00453,True


Variables numériques significatives :


,Variable,Moyenne_restés,Moyenne_partis,Différence,p_value,Significatif
0,Age,37.56,33.61,-3.95,0.00000,True
1,MonthlyIncome,6832.74,4787.09,-2045.65,0.00000,True
2,YearsAtCompany,7.37,5.13,-2.24,0.00000,True
4,TotalWorkingYears,11.86,8.24,-3.62,0.00000,True
7,YearsWithCurrManager,4.37,2.85,-1.52,0.00000,True
5,YearsInCurrentRole,4.48,2.90,-1.58,0.00000,True
3,DistanceFromHome,8.92,10.63,1.72,0.00414,True


In [12]:
output_dir = Path("../data/processed")

chi_square_results_df.to_csv(output_dir / "statistical_results_categorical.csv", index=False)
numeric_results_df.to_csv(output_dir / "statistical_results_numeric.csv", index=False)

print("Résultats statistiques sauvegardés avec succès.")

Résultats statistiques sauvegardés avec succès.


## Conclusion

Cette analyse statistique confirme que plusieurs variables sont associées à l'attrition des employés.

Les heures supplémentaires ressortent comme un facteur particulièrement important, avec un taux de départ beaucoup plus élevé chez les employés concernés.

L'ancienneté, l'âge, le salaire, le poste ou encore certains indicateurs de satisfaction peuvent également aider à identifier des profils plus exposés au turnover.

Ces résultats serviront de base pour la suite du projet : la construction d'un modèle prédictif capable d'estimer le risque de départ d'un employé.